# Training-data scaling (Figure 4C)

Top-1 classification accuracy of the `cell_dino` attention model (gene-KO classifier, C = 1,001) as a function of the per-set cell count, for four **training-set sizes** (1.5M → 50M cells). Larger training pools lift the whole accuracy curve; the x-axis is log-scaled so the early-bin gains (10 → 100 cells) stay visible.

Inputs are the per-training-size evaluation JSONs produced by the attention pipeline, committed alongside this notebook:

| file | training set |
| --- | --- |
| `attn_train1p5M.json` | 1.5M cells |
| `attn_train5M.json` | 5M cells |
| `attn_train15M.json` | 15M cells |
| `attn_train50M.json` | 50M cells |


## Imports

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.ticker import (
    FixedLocator,
    FuncFormatter,
    MultipleLocator,
    NullLocator,
    ScalarFormatter,
)

# Keep text editable in Illustrator (SVG keeps <text> elements rather than
# path-tracing the glyphs; PDF uses TrueType instead of Type-3).
plt.rcParams["svg.fonttype"] = "none"
plt.rcParams["pdf.fonttype"] = 42

FIGURES_DIR = Path("../../output/figure_4")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## Data paths

Per-training-size evaluation JSONs, committed alongside this notebook, ordered smallest → largest pool.

In [ ]:
FIGURE_DATA = Path(".")  # JSONs sit alongside this notebook

RUNS = [
    ("attn_train1p5M.json", "1.5M"),
    ("attn_train5M.json", "5M"),
    ("attn_train15M.json", "15M"),
    ("attn_train50M.json", "50M cells"),
]

## Configuration

Figure 4C uses **top-1** accuracy on a **log** x-axis, one curve per training-set size coloured by `viridis` (dark = largest pool). Flip `ACC_KEY` to `"mean_accuracy_top5"` for top-5.

In [ ]:
ACC_KEY = "mean_accuracy_top1"   # "mean_accuracy_top1" or "mean_accuracy_top5"

# viridis, ordered dark -> light for largest -> smallest training pool
COLORS = cm.viridis(np.linspace(0.9, 0.08, len(RUNS)))

## Load

Each JSON carries `n_cells_list` and the mean accuracy per bin (mean over the C classes). Assemble one tidy frame per run.

In [ ]:
def load(path: Path):
    d = json.loads(Path(path).read_text())
    df = pd.DataFrame({"n_cells": d["n_cells_list"], "acc": d[ACC_KEY]})
    return df, int(d["n_classes"])

runs = []
n_classes = None
for fname, label in RUNS:
    df, n_classes = load(FIGURE_DATA / fname)
    runs.append((label, df))

print(f"gene KO: {n_classes} classes | training sizes: {[l for l, _ in RUNS]}")

## Figure 4C

Single panel; log x-axis with explicit ticks at the actual cell-count slices (same treatment as Figure 4B). Saves an SVG (paper) + PNG.

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4.5))

all_x: set[int] = set()
for (label, df), color in zip(runs, COLORS):
    xs = df["n_cells"].to_numpy(dtype=float)
    ys = df["acc"].to_numpy(dtype=float)
    all_x.update(int(n) for n in df["n_cells"])
    ax.plot(xs, ys, "-o", color=color, linewidth=2.2, markersize=6,
            markeredgecolor="black", markeredgewidth=0.6, zorder=3, label=label)

ax.set_xlabel("# cells per bag", fontsize=11)
ax.set_ylabel("Classification accuracy", fontsize=11)
ax.set_title(f"Training-data scaling (gene KO, n={n_classes:,})", fontsize=12, fontweight="bold")
ax.tick_params(axis="both", labelsize=10)

# Fixed 0%-100% y-axis, tick every 10%; tiny headroom so ~100% markers don't clip.
ax.set_ylim(0.0, 1.03)
ax.yaxis.set_major_locator(MultipleLocator(0.1))
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{int(round(v * 100))}%"))
ax.grid(True, axis="y", alpha=0.25)

# Explicit ticks at the actual cell-count slices (log): set the scale first,
# then override the LogLocator with fixed ticks + a non-scientific formatter.
ticks = sorted(all_x)
ax.set_xscale("log")
ax.xaxis.set_major_locator(FixedLocator(ticks))
ax.xaxis.set_minor_locator(NullLocator())
fmt = ScalarFormatter()
fmt.set_scientific(False)
ax.xaxis.set_major_formatter(fmt)
ax.set_xlim(min(ticks) * 0.7, max(ticks) * 1.3)
ax.set_xticklabels([str(int(t)) for t in ticks])

# Largest training pool on top of the legend.
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[::-1], labels[::-1], loc="lower right", fontsize=9,
          framealpha=0.9, title="Training set size")

fig.tight_layout()
fig.savefig(FIGURES_DIR / "training_data_scaling.svg", bbox_inches="tight")
fig.savefig(FIGURES_DIR / "training_data_scaling.png", dpi=240, bbox_inches="tight")
plt.show()

## Summary table

Top-1 accuracy per (training set, n_cells), the values plotted above.

In [ ]:
rows = []
for label, df in runs:
    for _, r in df.iterrows():
        rows.append({"training_set": label, "n_cells": int(r["n_cells"]),
                     "top1_acc": float(r["acc"])})

summary = pd.DataFrame(rows)
summary.to_csv(FIGURES_DIR / "training_data_scaling_summary.csv", index=False)
summary